<a href="https://colab.research.google.com/github/gasortiz-hash/PichIA/blob/main/PichiIA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
# ================================================================
# PichIA CODON OPTIMIZATION FOR P. Pastoris
# ================================================================
import io
import os
import sys
import pickle
import datetime
import numpy as np
import ipywidgets as widgets
from IPython.display import display, HTML
from google.colab import drive
drive.mount('/content/drive')

# PDF Generation with ReportLab
try:
    from reportlab.lib.pagesizes import letter
    from reportlab.lib import colors
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, HRFlowable
except ImportError:
    !pip install reportlab --quiet
    from reportlab.lib.pagesizes import letter
    from reportlab.lib import colors
    from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
    from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Table, TableStyle, HRFlowable

# --- 1. CONFIGURATION AND KAZUSA TABLES ---
DRIVE_DIR = '/content/drive/MyDrive/PichIA'
MODEL_PATH = os.path.join(DRIVE_DIR, 'modelo_avanzado.pkl')
CODON_TO_INT_PATH = os.path.join(DRIVE_DIR, 'codon_to_int_avanzado.pkl')
INT_TO_CODON_PATH = os.path.join(DRIVE_DIR, 'int_to_codon_avanzado.pkl')

CODON_FRACTIONS_KAZUSA = {
    'GCT': 0.45, 'GCC': 0.25, 'GCA': 0.24, 'GCG': 0.06,
    'AGA': 0.47, 'CGT': 0.16, 'AGG': 0.16, 'CGA': 0.11, 'CGC': 0.05, 'CGG': 0.05,
    'AAC': 0.52, 'AAT': 0.48, 'GAT': 0.59, 'GAC': 0.41, 'TGT': 0.65, 'TGC': 0.35,
    'CAA': 0.62, 'CAG': 0.38, 'GAA': 0.58, 'GAG': 0.42, 'GGT': 0.43, 'GGA': 0.32,
    'GGC': 0.14, 'GGG': 0.10, 'CAT': 0.54, 'CAC': 0.46, 'ATT': 0.51, 'ATC': 0.31,
    'ATA': 0.18, 'TTG': 0.33, 'CTT': 0.17, 'CTG': 0.16, 'TTA': 0.15, 'CTA': 0.12,
    'CTC': 0.08, 'AAG': 0.53, 'AAA': 0.47, 'ATG': 1.00, 'TGG': 1.00, 'TTT': 0.56,
    'TTC': 0.44, 'CCA': 0.40, 'CCT': 0.35, 'CCC': 0.16, 'CCG': 0.10, 'TCT': 0.29,
    'TCC': 0.20, 'TCA': 0.19, 'AGT': 0.15, 'TCG': 0.09, 'AGC': 0.09, 'ACT': 0.40,
    'ACA': 0.25, 'ACC': 0.24, 'ACG': 0.11, 'TAC': 0.55, 'TAT': 0.45, 'GTT': 0.42,
    'GTC': 0.23, 'GTG': 0.20, 'GTA': 0.16, 'TAA': 0.53, 'TAG': 0.29, 'TGA': 0.18
}

CODON_TO_AA = {
    'TTT':'F','TTC':'F','TTA':'L','TTG':'L','CTT':'L','CTC':'L','CTA':'L','CTG':'L',
    'ATT':'I','ATC':'I','ATA':'I','ATG':'M','GTT':'V','GTC':'V','GTA':'V','GTG':'V',
    'TCT':'S','TCC':'S','TCA':'S','TCG':'S','CCT':'P','CCC':'P','CCA':'P','CCG':'P',
    'ACT':'T','ACC':'T','ACA':'T','ACG':'T','GCT':'A','GCC':'A','GCA':'A','GCG':'A',
    'TAT':'Y','TAC':'Y','CAT':'H','CAC':'H','CAA':'Q','CAG':'Q','AAT':'N','AAC':'N',
    'AAA':'K','AAG':'K','GAT':'D','GAC':'D','GAA':'E','GAG':'E','TGT':'C','TGC':'C',
    'TGG':'W','CGT':'R','CGC':'R','CGA':'R','CGG':'R','AGA':'R','AGG':'R','AGT':'S',
    'AGC':'S','GGT':'G','GGC':'G','GGA':'G','GGG':'G','TAA':'*','TAG':'*','TGA':'*'
}

F_MAX_MAP = {
    'F': 0.56, 'L': 0.33, 'S': 0.29, 'Y': 0.55, '*': 0.53,
    'C': 0.65, 'W': 1.00, 'P': 0.40, 'H': 0.54, 'Q': 0.62,
    'R': 0.47, 'I': 0.51, 'M': 1.00, 'T': 0.40, 'N': 0.52,
    'K': 0.53, 'V': 0.42, 'A': 0.45, 'D': 0.59, 'E': 0.58, 'G': 0.43
}

# --- 2. CALCULATION FUNCTIONS ---
def calcular_metricas(dna_seq):
    codons = [dna_seq[i:i+3] for i in range(0, len(dna_seq), 3)]
    if not codons:
        return 0.0, 0.0, 0.0

    gc_count = dna_seq.count('G') + dna_seq.count('C')
    gc_cont = gc_count / len(dna_seq)

    w_values = []
    rare_count = 0

    for c in codons:
        fraction = CODON_FRACTIONS_KAZUSA.get(c, 0.01)
        aa = CODON_TO_AA.get(c, 'X')
        max_f = F_MAX_MAP.get(aa, 1.0)

        w_i = fraction / max_f
        w_values.append(w_i)

        if fraction < 0.15:
            rare_count += 1

    cai = np.exp(np.mean(np.log(w_values))) if w_values else 0.0
    rare_frac = rare_count / len(codons)

    return cai, rare_frac, gc_cont

def cargar_modelo():
    if not (os.path.exists(MODEL_PATH) and os.path.exists(CODON_TO_INT_PATH) and os.path.exists(INT_TO_CODON_PATH)):
        print("❌ Error: Trained model files (.pkl) not found in Google Drive.")
        sys.exit(1)

    with open(MODEL_PATH, 'rb') as f:
        model = pickle.load(f)
    with open(CODON_TO_INT_PATH, 'rb') as f:
        codon_to_int = pickle.load(f)
    with open(INT_TO_CODON_PATH, 'rb') as f:
        int_to_codon = pickle.load(f)
    return model, codon_to_int, int_to_codon

def optimizar_codones(aa_seq, model, int_to_codon, window=5):
    all_aa = 'ACDEFGHIKLMNPQRSTVWY*'
    aa_to_idx = {aa: i for i, aa in enumerate(all_aa)}

    def encode_context(context_str):
        return [aa_to_idx.get(aa, 20) for aa in context_str]

    aa_seq = aa_seq.strip().upper()
    optimized_codons = []

    for i, aa in enumerate(aa_seq):
        context = []
        for j in range(i - window, i + window + 1):
            if j < 0 or j >= len(aa_seq):
                context.append('*')
            else:
                context.append(aa_seq[j])

        context_str = ''.join(context)
        pos = i / len(aa_seq) if len(aa_seq) > 1 else 0.5

        dna_parcial = ''.join(optimized_codons)
        gc_local = (dna_parcial.count('G') + dna_parcial.count('C')) / len(dna_parcial) if len(dna_parcial) > 0 else 0.45
        pair_score = 0.001

        features = np.array([encode_context(context_str) + [pos, gc_local, pair_score]])
        probs = model.predict_proba(features)[0]

        best_codon = None
        best_score = -1.0

        for idx, prob in enumerate(probs):
            codon = int_to_codon[idx]
            if CODON_TO_AA.get(codon) == aa:
                fraction = CODON_FRACTIONS_KAZUSA.get(codon, 0.0)
                max_f = F_MAX_MAP.get(aa, 1.0)
                w_i = fraction / max_f

                score = (prob * 0.3) + (w_i * 0.7)

                if score > best_score:
                    best_score = score
                    best_codon = codon

        optimized_codons.append(best_codon if best_codon else int_to_codon[np.argmax(probs)])

    return ''.join(optimized_codons)

# --- 3. PDF REPORT GENERATION (ENGLISH) ---
def generar_pdf_reporte(nombre_trabajo, aa_seq, dna_opt, cai, rare_frac, gc_cont):
    buffer = io.BytesIO()
    doc = SimpleDocTemplate(
        buffer, pagesize=letter, rightMargin=40, leftMargin=40, topMargin=40, bottomMargin=40
    )
    styles = getSampleStyleSheet()

    title_style = ParagraphStyle('Title', parent=styles['Heading1'], fontSize=18, textColor=colors.HexColor('#1A365D'), spaceAfter=10)
    section_style = ParagraphStyle('Sec', parent=styles['Heading2'], fontSize=12, textColor=colors.HexColor('#2B6CB0'), spaceBefore=10, spaceAfter=6)
    seq_style = ParagraphStyle('Seq', parent=styles['Code'], fontSize=8, leading=10, textColor=colors.HexColor('#2D3748'), wordWrap='CJK')

    elements = [
        Paragraph("Codon Optimization Report", title_style),
        Paragraph(f"<b>Project / Job Name:</b> {nombre_trabajo}", styles['Normal']),
        Paragraph(f"<b>Host Organism:</b> <i>Pichia pastoris</i> (Komagataella phaffii)", styles['Normal']),
        Paragraph(f"<b>Date:</b> {datetime.datetime.now().strftime('%Y-%m-%d %H:%M')}", styles['Normal']),
        Paragraph("Generated by PichiIA", styles['Normal']),
        Spacer(1, 10),
        HRFlowable(width="100%", thickness=1.5, color=colors.HexColor('#CBD5E0'), spaceAfter=15),
        Paragraph("📊 Quality Metrics Summary", section_style)
    ]

    data_metricas = [
        [Paragraph('<b>Metric</b>', styles['Normal']), Paragraph('<b>Calculated Value</b>', styles['Normal']), Paragraph('<b>Target Range</b>', styles['Normal'])],
        ['Codon Adaptation Index (CAI)', f"{cai:.4f}", '0.80 - 1.00 (Optimal)'],
        ['Low-Frequency Codons (<15%)', f"{rare_frac:.2%}", '< 5.00%'],
        ['Global GC Content', f"{gc_cont:.2%}", '35.00% - 50.00%']
    ]

    t_metricas = Table(data_metricas, colWidths=[200, 140, 180])
    t_metricas.setStyle(TableStyle([
        ('BACKGROUND', (0, 0), (-1, 0), colors.HexColor('#E2E8F0')),
        ('GRID', (0, 0), (-1, -1), 0.5, colors.HexColor('#CBD5E0')),
        ('PADDING', (0, 0), (-1, -1), 6),
    ]))
    elements.extend([t_metricas, Spacer(1, 15)])
    elements.extend([Paragraph(f"🔬 Original AA Sequence ({len(aa_seq)} aa)", section_style), Paragraph(aa_seq, seq_style), Spacer(1, 10)])
    elements.extend([Paragraph(f"🧬 Optimized DNA Sequence ({len(dna_opt)} bp)", section_style), Paragraph(dna_opt, seq_style)])

    doc.build(elements)
    buffer.seek(0)
    return buffer.getvalue()

# --- 4. LOAD MODEL AND INITIALIZE WIDGETS ---
model, codon_to_int, int_to_codon = cargar_modelo()

header_html = HTML("""
<div style="background-color: #2B6CB0; padding: 12px; border-radius: 8px; color: white; margin-bottom: 12px;">
    <h3 style="margin: 0; color: white;">🧬 PichIA - Codon Optimization Tool (Pichia pastoris)</h3>
    <p style="margin: 3px 0 0 0; opacity: 0.9; font-size: 13px;">Optimize amino acid sequences, preview sequence alignment, and download output files</p>
</div>
""")

txt_nombre = widgets.Text(
    value='Optimized_Protein_Project',
    description='Project Name:',
    placeholder='Enter project or job name',
    layout=widgets.Layout(width='98%')
)

txt_secuencia = widgets.Textarea(
    value='Your AA seq.',
    description='AA Sequence:',
    placeholder='Paste your amino acid sequence here...',
    layout=widgets.Layout(width='98%', height='130px')
)

btn_procesar = widgets.Button(
    description='⚡ Optimize Codons',
    button_style='primary',
    icon='cogs',
    layout=widgets.Layout(width='200px', height='38px')
)

out_resultados = widgets.Output()

# Temporary storage for files
archivos_generados = {}

def ejecutar_optimizacion_widget(b):
    out_resultados.clear_output()

    aa_seq = txt_secuencia.value.strip().upper()
    if aa_seq.startswith('>'):
        aa_seq = ''.join(aa_seq.split('\n')[1:])
    aa_seq = ''.join(aa_seq.split())

    nombre_trabajo = txt_nombre.value.strip() or "Optimization_Result"

    if not aa_seq:
        with out_resultados:
            print("❌ Please enter a valid amino acid sequence.")
        return

    with out_resultados:
        print("⏳ Processing optimization with PichIA...")
        dna_opt = optimizar_codones(aa_seq, model, int_to_codon)
        cai, rare_frac, gc_cont = calcular_metricas(dna_opt)

        out_resultados.clear_output()

        # FASTA Format
        fasta_content = f">{nombre_trabajo}_Pichia_Optimized | Length: {len(dna_opt)}bp | CAI: {cai:.4f}\n"
        for i in range(0, len(dna_opt), 80):
            fasta_content += dna_opt[i:i+80] + "\n"

        # PDF Report
        pdf_bytes = generar_pdf_reporte(nombre_trabajo, aa_seq, dna_opt, cai, rare_frac, gc_cont)

        # Save temp files for Colab native download
        fasta_path = f"/content/{nombre_trabajo}_optimized.fasta"
        pdf_path = f"/content/{nombre_trabajo}_report.pdf"

        with open(fasta_path, "w") as f:
            f.write(fasta_content)
        with open(pdf_path, "wb") as f:
            f.write(pdf_bytes)

        archivos_generados['fasta'] = fasta_path
        archivos_generados['pdf'] = pdf_path

        # HTML Results Box
        html_res = f"""
        <div style="background-color: #F7FAFC; border: 1px solid #CBD5E0; padding: 15px; border-radius: 8px; margin-top: 10px;">
            <h4 style="color: #2D3748; margin-top: 0; margin-bottom: 10px;">📊 Optimization Results Summary:</h4>
            <ul style="margin: 0 0 15px 0; padding-left: 20px; font-size: 14px; color: #2D3748;">
                <li><b>CAI (Codon Adaptation Index):</b> <span style="color:#2B6CB0;"><b>{cai:.4f}</b></span></li>
                <li><b>Low-frequency codons (&lt;15%):</b> <span style="color:#38A169;"><b>{rare_frac:.2%}</b></span></li>
                <li><b>Global GC Content:</b> <b>{gc_cont:.2%}</b></li>
            </ul>

            <label style="font-weight: bold; color: #2D3748; font-size: 13px;">🔬 Original Amino Acid Sequence ({len(aa_seq)} aa):</label>
            <textarea readonly style="width: 100%; height: 70px; font-family: monospace; font-size: 11px; margin-top: 4px; margin-bottom: 12px; border: 1px solid #CBD5E0; border-radius: 4px; padding: 6px; background-color: #FFFFFF;">{aa_seq}</textarea>

            <label style="font-weight: bold; color: #2B6CB0; font-size: 13px;">🧬 Optimized DNA Sequence ({len(dna_opt)} bp):</label>
            <textarea readonly style="width: 100%; height: 90px; font-family: monospace; font-size: 11px; margin-top: 4px; margin-bottom: 10px; border: 1px solid #CBD5E0; border-radius: 4px; padding: 6px; background-color: #FFFFFF;">{dna_opt}</textarea>
        </div>
        """
        display(HTML(html_res))

        # Download buttons
        btn_dl_fasta = widgets.Button(description='📥 Download FASTA', button_style='success', icon='file-code')
        btn_dl_pdf = widgets.Button(description='📄 Download PDF Report', button_style='info', icon='file-pdf')

        def descargar_fasta_action(b):
            files.download(archivos_generados['fasta'])

        def descargar_pdf_action(b):
            files.download(archivos_generados['pdf'])

        btn_dl_fasta.on_click(descargar_fasta_action)
        btn_dl_pdf.on_click(descargar_pdf_action)

        display(widgets.HBox([btn_dl_fasta, btn_dl_pdf], layout=widgets.Layout(margin='10px 0 0 0')))

btn_procesar.on_click(ejecutar_optimizacion_widget)

# Render widgets
display(header_html)
display(widgets.VBox([
    txt_nombre,
    txt_secuencia,
    btn_procesar,
    out_resultados
]))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
